In [10]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sharooqfarzeenak/real-life-deception-detection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'real-life-deception-detection-dataset' dataset.
Path to dataset files: /kaggle/input/real-life-deception-detection-dataset


In [11]:
import os

dataset_path = path  # from kagglehub.dataset_download()

for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}📁 {os.path.basename(root)}")

    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}📄 {f}")


📁 real-life-deception-detection-dataset
    📁 Real-life Deception Detection Dataset With Train Test
        📁 Test
            📄 trial_truth_056.mp4
            📄 trial_lie_060.mp4
            📄 trial_truth_057.mp4
            📄 trial_lie_057.mp4
            📄 trial_truth_060.mp4
            📄 trial_lie_056.mp4
            📄 trial_truth_055.mp4
            📄 trial_truth_058.mp4
            📄 trial_lie_059.mp4
            📄 trial_truth_059.mp4
            📄 trial_lie_061.mp4
            📄 trial_lie_058.mp4
        📁 Train
            📄 trial_lie_028.mp4
            📄 trial_lie_019.mp4
            📄 trial_lie_038.mp4
            📄 trial_truth_009.mp4
            📄 trial_truth_011.mp4
            📄 trial_truth_019.mp4
            📄 trial_truth_047.mp4
            📄 trial_lie_049.mp4
            📄 trial_lie_008.mp4
            📄 trial_truth_032.mp4
            📄 trial_lie_036.mp4
            📄 trial_lie_014.mp4
            📄 trial_lie_042.mp4
            📄 trial_lie_037.mp4
            📄 t

In [12]:
import os
import cv2
import librosa
import numpy as np
from moviepy.editor import VideoFileClip
from tqdm import tqdm

# ==============================
# PATH CONFIG
# ==============================
INPUT_ROOT = path  # kagglehub downloaded path

DATASET_ROOT = os.path.join(
    INPUT_ROOT,
    "Real-life Deception Detection Dataset With Train Test"
)

OUTPUT_ROOT = "/content/lie_preprocessed_data"

FRAME_SIZE = (224, 224)
MAX_FRAMES = 20
FRAME_INTERVAL = 15
MFCC_FEATURES = 40

# ==============================
# CREATE DIRECTORY STRUCTURE
# ==============================
def create_dirs():
    for modality in ["face", "audio"]:
        for split in ["train", "test"]:
            for label in ["lie", "truth"]:
                os.makedirs(
                    os.path.join(OUTPUT_ROOT, modality, split, label),
                    exist_ok=True
                )

create_dirs()

# ==============================
# FRAME EXTRACTION
# ==============================
def extract_frames(video_path, save_dir):
    cap = cv2.VideoCapture(video_path)
    frame_count = 0
    saved = 0

    while cap.isOpened() and saved < MAX_FRAMES:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % FRAME_INTERVAL == 0:
            frame = cv2.resize(frame, FRAME_SIZE)
            cv2.imwrite(
                os.path.join(save_dir, f"frame_{saved}.jpg"),
                frame
            )
            saved += 1

        frame_count += 1

    cap.release()

# ==============================
# AUDIO → MFCC EXTRACTION
# ==============================
def extract_mfcc(video_path, save_path):
    try:
        video = VideoFileClip(video_path)
        audio = video.audio
        wav_path = save_path.replace(".npy", ".wav")

        audio.write_audiofile(
            wav_path,
            fps=16000,
            verbose=False,
            logger=None
        )

        y, sr = librosa.load(wav_path, sr=16000)
        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=MFCC_FEATURES
        )
        mfcc = np.mean(mfcc.T, axis=0)

        np.save(save_path, mfcc)
        os.remove(wav_path)

    except Exception as e:
        print("Audio error:", video_path, e)

# ==============================
# MAIN LOOP
# ==============================
for split in ["Train", "Test"]:
    split_lower = split.lower()
    split_path = os.path.join(DATASET_ROOT, split)

    print(f"\nProcessing {split} data...")

    for video_file in tqdm(os.listdir(split_path)):
        if not video_file.endswith(".mp4"):
            continue

        video_path = os.path.join(split_path, video_file)

        label = "lie" if "lie" in video_file.lower() else "truth"

        # ----- FACE FRAMES -----
        face_save_dir = os.path.join(
            OUTPUT_ROOT,
            "face",
            split_lower,
            label,
            video_file.replace(".mp4", "")
        )
        os.makedirs(face_save_dir, exist_ok=True)
        extract_frames(video_path, face_save_dir)

        # ----- AUDIO MFCC -----
        audio_save_path = os.path.join(
            OUTPUT_ROOT,
            "audio",
            split_lower,
            label,
            video_file.replace(".mp4", ".npy")
        )
        extract_mfcc(video_path, audio_save_path)

print("\n✅ PREPROCESSING COMPLETE!")
print("📁 Saved at:", OUTPUT_ROOT)



Processing Train data...


100%|██████████| 109/109 [02:30<00:00,  1.38s/it]



Processing Test data...


100%|██████████| 12/12 [00:08<00:00,  1.39it/s]


✅ PREPROCESSING COMPLETE!
📁 Saved at: /content/lie_preprocessed_data


In [13]:
import os
import shutil

SOURCE_DIR = "/content/lie_preprocessed_data"
DEST_DIR = "/content/drive/MyDrive/lie_preprocessed_data"


In [14]:
# Remove old copy if exists (prevents overwrite issues)
if os.path.exists(DEST_DIR):
    shutil.rmtree(DEST_DIR)

shutil.copytree(SOURCE_DIR, DEST_DIR)

print("✅ Preprocessed data successfully saved to Google Drive!")
print("📁 Location:", DEST_DIR)


✅ Preprocessed data successfully saved to Google Drive!
📁 Location: /content/drive/MyDrive/lie_preprocessed_data


In [15]:
import os
import shutil

BASE_DIR = "/content/drive/MyDrive/lie_preprocessed_data/face"

def flatten_dataset(split):
    for label in ["lie", "truth"]:
        label_dir = os.path.join(BASE_DIR, split, label)
        for sub in os.listdir(label_dir):
            sub_path = os.path.join(label_dir, sub)
            if os.path.isdir(sub_path):
                for img in os.listdir(sub_path):
                    shutil.move(
                        os.path.join(sub_path, img),
                        os.path.join(label_dir, img)
                    )
                os.rmdir(sub_path)

flatten_dataset("train")
flatten_dataset("test")

print("✅ Dataset flattened successfully")


✅ Dataset flattened successfully


In [17]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


IMG_SIZE = 224
BATCH_SIZE = 32

train_dir = BASE_DIR + "/train"
test_dir  = BASE_DIR + "/test"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_data.class_indices)





Found 40 images belonging to 2 classes.
Found 40 images belonging to 2 classes.
Class mapping: {'lie': 0, 'truth': 1}


In [18]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # transfer learning

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [19]:
EPOCHS = 15

history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS
)


  self._warn_if_super_not_called()



Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 21s 9s/step - accuracy: 0.4354 - loss: 1.0149 - val_accuracy: 0.6500 - val_loss: 0.7172
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - accuracy: 0.5437 - loss: 0.7797 - val_accuracy: 0.6000 - val_loss: 0.7086
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - accuracy: 0.7417 - loss: 0.5492 - val_accuracy: 0.4000 - val_loss: 0.7139
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 3s/step - accuracy: 0.6958 - loss: 0.4990 - val_accuracy: 0.3750 - val_loss: 0.7250
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - accuracy: 0.7500 - loss: 0.5010 - val_accuracy: 0.3000 - val_loss: 0.7427
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.9021 - loss: 0.3530 - val_accuracy: 0.3250 - val_loss: 0.7627
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 6s/step - accuracy: 0.8313 - loss: 0.3838 - val_accuracy: 0.3250 - val_loss: 0.7838
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 1.0000 - loss: 0.2074 - val_accuracy: 0.3250 - val_loss: 0.8042
Epoch 9/15
2/2 

In [20]:
base_model.trainable = True

# Freeze early layers
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_data,
    validation_data=test_data,
    epochs=10
)


Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 22s 7s/step - accuracy: 0.6583 - loss: 0.6212 - val_accuracy: 0.2750 - val_loss: 0.9070
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 3s/step - accuracy: 0.5833 - loss: 0.5823 - val_accuracy: 0.2750 - val_loss: 0.9131
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 8s 6s/step - accuracy: 0.7771 - loss: 0.4928 - val_accuracy: 0.2750 - val_loss: 0.9190
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.8917 - loss: 0.3848 - val_accuracy: 0.2750 - val_loss: 0.9242
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - accuracy: 0.8750 - loss: 0.4003 - val_accuracy: 0.2750 - val_loss: 0.9306
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 6s 3s/step - accuracy: 0.8917 - loss: 0.3211 - val_accuracy: 0.2750 - val_loss: 0.9371
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - accuracy: 0.8000 - loss: 0.4130 - val_accuracy: 0.2500 - val_loss: 0.9451
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 3s/step - accuracy: 0.9500 - loss: 0.2981 - val_accuracy: 0.2500 - val_loss: 0.9527
Epoch 9/10
2/2 

In [21]:
MODEL_PATH = "/content/drive/MyDrive/mobilenet_face_lie_detection.h5"
model.save(MODEL_PATH)

print("✅ Model saved at:", MODEL_PATH)


✅ Model saved at: /content/drive/MyDrive/mobilenet_face_lie_detection.h5


In [33]:
# =====================================
# FACE IMAGE TESTING – MOBILE NET MODEL
# =====================================

import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model

# -------------------------------------
# CONFIG
# -------------------------------------
MODEL_PATH = "/content/drive/MyDrive/mobilenet_face_lie_detection.h5"
IMG_SIZE = 224

# -------------------------------------
# LOAD TRAINED MODEL
# -------------------------------------
model = load_model(MODEL_PATH)
print("✅ Face MobileNet model loaded successfully")

# -------------------------------------
# IMAGE PREDICTION FUNCTION
# -------------------------------------
def predict_image(img_path):
    """
    Predict lie/truth from a single face image
    """
    img = image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img = image.img_to_array(img) / 255.0
    img = np.expand_dims(img, axis=0)

    prob = model.predict(img, verbose=0)[0][0]

    if prob > 0.5:
        return {
            "prediction": "Lie",
            "confidence": round(prob * 100, 2)
        }
    else:
        return {
            "prediction": "Truth",
            "confidence": round((1 - prob) * 100, 2)
        }

# -------------------------------------
# TEST IMAGE
# -------------------------------------
test_image_path = "/content/drive/MyDrive/processed_dataset/face/Test/truth/trial_truth_056/frame_0.jpg"

result = predict_image(test_image_path)

print("\n🧠 FACE DECEPTION RESULT")
print("------------------------")
print("Prediction :", result["prediction"])
print("Confidence :", result["confidence"], "%")


✅ Face MobileNet model loaded successfully

🧠 FACE DECEPTION RESULT
------------------------
Prediction : Truth
Confidence : 63.38 %


In [25]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
)
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle

BASE_DIR = "/content/drive/MyDrive/lie_preprocessed_data/audio"

def load_audio_data(split):
    X, y = [], []

    for label in ["truth", "lie"]:
        label_dir = os.path.join(BASE_DIR, split, label)
        for file in os.listdir(label_dir):
            if file.endswith(".npy"):
                mfcc = np.load(os.path.join(label_dir, file))
                X.append(mfcc)
                y.append(label)

    X = np.array(X)
    y = np.array(y)

    return X, y

X_train, y_train = load_audio_data("train")
X_test, y_test   = load_audio_data("test")

# Encode labels: truth=0, lie=1
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test  = encoder.transform(y_test)

# Shuffle
X_train, y_train = shuffle(X_train, y_train, random_state=42)

# Reshape for CNN (samples, features, channels)
X_train = X_train[..., np.newaxis]
X_test  = X_test[..., np.newaxis]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)



Train shape: (109, 40, 1)
Test shape : (12, 40, 1)


In [26]:
model = Sequential([

    Conv1D(64, kernel_size=3, activation='relu', input_shape=(40,1)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(256, kernel_size=3, activation='relu'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 38, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 38, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 19, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 17, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 17, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 8, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 6, 256)         │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 6, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 3, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        98,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 223,873 (874.50 KB)

 Trainable params: 222,977 (871.00 KB)

 Non-trainable params: 896 (3.50 KB)

In [27]:
EPOCHS = 30
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)


Epoch 1/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 139ms/step - accuracy: 0.5457 - loss: 1.3487 - val_accuracy: 0.5000 - val_loss: 0.9223
Epoch 2/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.6272 - loss: 0.7725 - val_accuracy: 0.5000 - val_loss: 0.7723
Epoch 3/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.6794 - loss: 0.6199 - val_accuracy: 0.5000 - val_loss: 0.7528
Epoch 4/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.7175 - loss: 0.7294 - val_accuracy: 0.5000 - val_loss: 0.7456
Epoch 5/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.7201 - loss: 0.5917 - val_accuracy: 0.5000 - val_loss: 0.7503
Epoch 6/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7854 - loss: 0.5080 - val_accuracy: 0.5000 - val_loss: 0.7534
Epoch 7/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8281 - loss: 0.4627 - val_accuracy: 0.5000 - val_loss: 0.7589
Epoch 8/30
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8068 - loss: 0.4114 - val_accuracy: 0.5000 - val_loss: 0.7580

In [28]:
AUDIO_MODEL_PATH = "/content/drive/MyDrive/audio_cnn_lie_detection.h5"
model.save(AUDIO_MODEL_PATH)

print("✅ Audio CNN model saved at:", AUDIO_MODEL_PATH)


✅ Audio CNN model saved at: /content/drive/MyDrive/audio_cnn_lie_detection.h5


In [32]:
# ==========================================
# AUDIO CNN TESTING (MP4 OR WAV SUPPORTED)
# ==========================================

import os
import numpy as np
import librosa
from moviepy.editor import VideoFileClip
from tensorflow.keras.models import load_model

# ------------------------------------------
# CONFIG
# ------------------------------------------
AUDIO_MODEL_PATH = "/content/drive/MyDrive/audio_cnn_lie_detection.h5"
N_MFCC = 40
SR = 16000

# ------------------------------------------
# LOAD MODEL
# ------------------------------------------
audio_model = load_model(AUDIO_MODEL_PATH)
print("✅ Audio CNN model loaded successfully")

# ------------------------------------------
# AUDIO → MFCC FUNCTION (ROBUST)
# ------------------------------------------
def audio_to_mfcc(input_path):
    """

    - .wav (audio)

    Output MFCC shape: (40,)
    """

    ext = os.path.splitext(input_path)[1].lower()


    if ext == ".wav":
        y, sr = librosa.load(input_path, sr=SR)

    else:
        raise ValueError("Unsupported file format. Use .mp4 or .wav")

    # -------- MFCC EXTRACTION --------
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=N_MFCC
    )

    # Mean over time (MUST match training)
    mfcc = np.mean(mfcc.T, axis=0)

    return mfcc

# ------------------------------------------
# PREDICTION FUNCTION
# ------------------------------------------
def predict_audio(input_path):
    mfcc = audio_to_mfcc(input_path)

    # CNN input shape: (1, 40, 1)
    mfcc = mfcc.reshape(1, N_MFCC, 1)

    prob = audio_model.predict(mfcc, verbose=0)[0][0]

    if prob > 0.5:
        return {
            "prediction": "Lie",
            "confidence": round(prob * 100, 2)
        }
    else:
        return {
            "prediction": "Truth",
            "confidence": round((1 - prob) * 100, 2)
        }

# ------------------------------------------
# TEST
# ------------------------------------------
test_file = "/content/drive/MyDrive/processed_dataset/audio/Test/lie/trial_lie_056.wav"  # WAV

result = predict_audio(test_file)

print("\n🎤 AUDIO DECEPTION RESULT")
print("-------------------------")
print("Prediction :", result["prediction"])
print("Confidence :", result["confidence"], "%")


✅ Audio CNN model loaded successfully

🎤 AUDIO DECEPTION RESULT
-------------------------
Prediction : Lie
Confidence : 68.66 %
